# VIXY Best Model Builder v2
**Goal**: Find the best model for live daily VIXY T+2 direction predictions.

**Experiments vs v1:**
- XGBoost (new)
- LightGBM (better hyperparams than v1)
- Soft-voting ensemble of top configs
- Rolling 3-year training window
- Recency-weighted training
- Calibrated probabilities

All evaluated with identical walk-forward 2019–2025 framework for fair comparison.

In [ ]:
# ============================================================
# CELL 1: IMPORTS & CONFIG
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import joblib
from pathlib import Path
from datetime import datetime, timedelta

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (VotingClassifier, HistGradientBoostingClassifier,
                               GradientBoostingClassifier)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
    print('XGBoost available')
except ImportError:
    HAS_XGB = False
    print('XGBoost NOT available - run: pip install xgboost')

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
    print('LightGBM available')
except ImportError:
    HAS_LGBM = False
    print('LightGBM NOT available - run: pip install lightgbm')

# ---- PATHS ----
FEATS_PATH   = 'D:/work/VIXY model/feats.csv'
OUTPUT_DIR   = Path('D:/work/VIXY model/vixy_best_model_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- CONSTANTS (must match v1 for fair comparison) ----
TICKER               = 'VIXY'
EXTRA_MARKET_LAG     = 1       # VIXY-derived features shifted by 1 extra day
TARGET_FORWARD_DAYS  = 2       # predict T+2 direction
RETURN_THRESHOLD     = 0.0     # binary: up if ret > 0

# Best config found in v1 (Cell 4 winner)
BEST_LAGS  = [1, 2, 4, 6, 10, 15]
BEST_ROLLS = [2, 3, 5, 8, 10]
BEST_KBEST = 120

# Walk-forward test years
TEST_YEARS = list(range(2019, 2026))

print(f'Output dir: {OUTPUT_DIR}')
print('Config loaded OK')

In [ ]:
# ============================================================
# CELL 2: DATA LOADING
# ============================================================
def load_data():
    # Load macro features
    feats = pd.read_csv(FEATS_PATH, parse_dates=['date'])
    feats = feats.sort_values('date').reset_index(drop=True)
    print(f'Features: {feats.shape}  ({feats.date.min().date()} to {feats.date.max().date()})')

    # Download VIXY from Yahoo Finance
    start = feats['date'].min().strftime('%Y-%m-%d')
    end   = (datetime.today() + timedelta(days=2)).strftime('%Y-%m-%d')
    vixy  = yf.download(TICKER, start=start, end=end, progress=False, auto_adjust=False)

    if isinstance(vixy.columns, pd.MultiIndex):
        vixy.columns = [c[0].lower() for c in vixy.columns]
    else:
        vixy.columns = [c.lower() for c in vixy.columns]

    vixy = vixy[['close']].rename(columns={'close': 'vixy_close'})
    vixy['vixy_ret']     = vixy['vixy_close'].pct_change()
    vixy['vixy_log_ret'] = np.log(vixy['vixy_close'] / vixy['vixy_close'].shift(1))
    vixy.index.name = 'date'
    vixy = vixy.reset_index()
    vixy['date'] = pd.to_datetime(vixy['date'])

    df = feats.merge(vixy, on='date', how='inner')
    df = df.sort_values('date').reset_index(drop=True)
    print(f'Merged:   {df.shape}  ({df.date.min().date()} to {df.date.max().date()})')
    return df

raw_df = load_data()

In [ ]:
# ============================================================
# CELL 3: FEATURE ENGINEERING  (identical to v1 — no look-ahead)
# ============================================================
def add_vixy_features_decision_safe(df, lag_set, rolling_windows,
                                     extra_market_lag=EXTRA_MARKET_LAG,
                                     target_forward_days=TARGET_FORWARD_DAYS):
    out = df.copy()
    for lag in lag_set:
        safe_lag = lag + extra_market_lag
        out[f'ret_lag_{lag}']    = out['vixy_ret'].shift(safe_lag)
        out[f'logret_lag_{lag}'] = out['vixy_log_ret'].shift(safe_lag)
        out[f'close_lag_{lag}']  = out['vixy_close'].shift(safe_lag)

    for w in rolling_windows:
        out[f'ret_mean_{w}']    = out['vixy_ret'].rolling(w).mean().shift(extra_market_lag)
        out[f'ret_std_{w}']     = out['vixy_ret'].rolling(w).std().shift(extra_market_lag)
        out[f'price_ma_{w}']    = out['vixy_close'].rolling(w).mean().shift(extra_market_lag)
        out[f'price_vs_ma_{w}'] = (out['vixy_close'] / out['vixy_close'].rolling(w).mean() - 1
                                   ).shift(extra_market_lag)
        out[f'cumret_{w}']      = ((1 + out['vixy_ret']).rolling(w)
                                   .apply(np.prod, raw=True) - 1).shift(extra_market_lag)

    out['target_ret_fwd'] = out['vixy_ret'].shift(-target_forward_days)
    out['target_up']      = (out['target_ret_fwd'] > RETURN_THRESHOLD).astype(int)
    return out


def get_feature_cols(df, feature_mode):
    macro_cols   = [c for c in df.columns if c.startswith('F') and c[1:].isdigit()]
    vixy_ar_cols = [c for c in df.columns if c.startswith((
        'ret_lag_', 'logret_lag_', 'close_lag_',
        'ret_mean_', 'ret_std_', 'price_ma_', 'price_vs_ma_', 'cumret_'
    ))]
    if feature_mode == 'macro_only':
        return macro_cols
    elif feature_mode == 'macro_plus_ar':
        ar_cols = [c for c in vixy_ar_cols if c.startswith(('ret_lag_', 'logret_lag_', 'ret_mean_', 'ret_std_'))]
        return macro_cols + ar_cols
    elif feature_mode == 'macro_plus_all_vixy':
        return macro_cols + vixy_ar_cols
    else:
        raise ValueError(f'Unknown feature_mode: {feature_mode}')


# Build engineered dataset with best lag/roll config
df_eng = add_vixy_features_decision_safe(raw_df, BEST_LAGS, BEST_ROLLS)

feat_cols_all_vixy = get_feature_cols(df_eng, 'macro_plus_all_vixy')
feat_cols_ar       = get_feature_cols(df_eng, 'macro_plus_ar')
feat_cols_macro    = get_feature_cols(df_eng, 'macro_only')

print(f'macro_only:          {len(feat_cols_macro)} features')
print(f'macro_plus_ar:       {len(feat_cols_ar)} features')
print(f'macro_plus_all_vixy: {len(feat_cols_all_vixy)} features')
print(f'Target class balance: {df_eng["target_up"].value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
# ============================================================
# CELL 4: WALK-FORWARD EVALUATION FRAMEWORK
# ============================================================
def build_pipeline(model, feature_cols, kbest=BEST_KBEST, use_scaler=True):
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if use_scaler:
        steps.append(('scaler', StandardScaler()))
    if kbest != 'all' and kbest < len(feature_cols):
        steps.append(('select', SelectKBest(score_func=mutual_info_classif, k=kbest)))
    steps.append(('model', model))
    return Pipeline(steps)


def walk_forward(df_feats, model_factory, feature_cols,
                 test_years=TEST_YEARS, threshold=0.5,
                 kbest=BEST_KBEST, use_scaler=True,
                 rolling_years=None,
                 sample_weight_fn=None,
                 return_preds=False):
    """
    Expanding window (rolling_years=None) or rolling window walk-forward.
    sample_weight_fn: callable(train_df) -> array of weights
    """
    results, all_preds = [], []

    for year in test_years:
        if rolling_years is None:
            train_mask = df_feats['date'].dt.year < year
        else:
            train_mask = ((df_feats['date'].dt.year >= year - rolling_years) &
                          (df_feats['date'].dt.year < year))
        test_mask = df_feats['date'].dt.year == year

        req_cols = feature_cols + ['target_up']
        train_df = df_feats[train_mask].dropna(subset=req_cols)
        test_df  = df_feats[test_mask].dropna(subset=req_cols)

        if len(train_df) < 200 or len(test_df) < 20:
            continue

        X_tr, y_tr = train_df[feature_cols].values, train_df['target_up'].values
        X_te, y_te = test_df[feature_cols].values,  test_df['target_up'].values

        pipe = build_pipeline(model_factory(), feature_cols, kbest, use_scaler)

        if sample_weight_fn is not None:
            sw = sample_weight_fn(train_df)
            try:
                pipe.fit(X_tr, y_tr, **{'model__sample_weight': sw})
            except Exception:
                pipe.fit(X_tr, y_tr)
        else:
            pipe.fit(X_tr, y_tr)

        probs = pipe.predict_proba(X_te)[:, 1]
        preds = (probs >= threshold).astype(int)

        acc     = accuracy_score(y_te, preds)
        bal_acc = balanced_accuracy_score(y_te, preds)
        try:
            auc = roc_auc_score(y_te, probs)
        except Exception:
            auc = np.nan
        f1 = f1_score(y_te, preds, zero_division=0)

        results.append({'year': year, 'accuracy': acc, 'balanced_accuracy': bal_acc,
                        'roc_auc': auc, 'f1': f1,
                        'n_train': len(train_df), 'n_test': len(test_df)})
        if return_preds:
            p = test_df[['date', 'target_up']].copy()
            p['prob_up'] = probs
            p['pred']    = preds
            p['year']    = year
            all_preds.append(p)

    res_df   = pd.DataFrame(results)
    preds_df = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()
    return (res_df, preds_df) if return_preds else res_df


def summarise(name, res_df):
    r = res_df
    print(f"  acc={r.accuracy.mean():.4f}  bal_acc={r.balanced_accuracy.mean():.4f}"
          f"  auc={r.roc_auc.mean():.4f}  f1={r.f1.mean():.4f}"
          f"  acc_min={r.accuracy.min():.4f}  << {name}")
    return {
        'experiment': name,
        'acc_mean':      round(r.accuracy.mean(), 5),
        'bal_acc_mean':  round(r.balanced_accuracy.mean(), 5),
        'roc_auc_mean':  round(r.roc_auc.mean(), 5),
        'f1_mean':       round(r.f1.mean(), 5),
        'acc_min':       round(r.accuracy.min(), 5),
        'bal_acc_std':   round(r.balanced_accuracy.std(), 5),
    }

print('Framework ready.')

In [ ]:
# ============================================================
# CELL 5: EXPERIMENTS
# Run time: ~45-90 minutes total
# ============================================================
all_summaries = []
all_res       = {}   # store detailed yearly results for each exp

# ----------------------------------------------------------------
# EXP A: BASELINE — replicate v1 best (LogReg L2 C=0.5, k=120)
# ----------------------------------------------------------------
print('=' * 60)
print('EXP A: Baseline  LogReg L2 C=0.5  macro_plus_all_vixy  k=120  thr=0.56')
res = walk_forward(
    df_eng,
    lambda: LogisticRegression(penalty='l2', C=0.5, solver='saga', max_iter=2000, random_state=42),
    feat_cols_all_vixy, threshold=0.56, kbest=120
)
all_summaries.append(summarise('A_baseline_logreg_l2_C0.5', res))
all_res['A_baseline'] = res
print(res[['year','accuracy','balanced_accuracy','roc_auc','f1']].to_string(index=False))

# ----------------------------------------------------------------
# EXP B: Vary threshold on baseline (find optimal)
# ----------------------------------------------------------------
print('\n' + '=' * 60)
print('EXP B: Threshold sweep on baseline')
for thr in [0.50, 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57, 0.58]:
    r = walk_forward(
        df_eng,
        lambda: LogisticRegression(penalty='l2', C=0.5, solver='saga', max_iter=2000, random_state=42),
        feat_cols_all_vixy, threshold=thr, kbest=120
    )
    all_summaries.append(summarise(f'B_logreg_thr{thr}', r))
    all_res[f'B_thr{thr}'] = r

# ----------------------------------------------------------------
# EXP C: LogReg with different C values (fine-grained around best)
# ----------------------------------------------------------------
print('\n' + '=' * 60)
print('EXP C: LogReg C sweep  (l2, all_vixy, k=120, thr=0.53)')
for C in [0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0]:
    r = walk_forward(
        df_eng,
        lambda C=C: LogisticRegression(penalty='l2', C=C, solver='saga', max_iter=2000, random_state=42),
        feat_cols_all_vixy, threshold=0.53, kbest=120
    )
    all_summaries.append(summarise(f'C_logreg_l2_C{C}', r))
    all_res[f'C_l2_C{C}'] = r

# ----------------------------------------------------------------
# EXP D: HistGradientBoosting (built into sklearn, no extra install)
# ----------------------------------------------------------------
print('\n' + '=' * 60)
print('EXP D: HistGradientBoosting')
hgb_configs = [
    dict(max_iter=200, max_depth=4, learning_rate=0.05, l2_regularization=1.0, min_samples_leaf=20),
    dict(max_iter=300, max_depth=3, learning_rate=0.03, l2_regularization=2.0, min_samples_leaf=30),
    dict(max_iter=400, max_depth=5, learning_rate=0.05, l2_regularization=0.5, min_samples_leaf=15),
    dict(max_iter=200, max_depth=4, learning_rate=0.05, l2_regularization=1.0, min_samples_leaf=20,
         class_weight='balanced'),
]
for i, cfg in enumerate(hgb_configs, 1):
    r = walk_forward(
        df_eng,
        lambda cfg=cfg: HistGradientBoostingClassifier(**cfg, random_state=42),
        feat_cols_all_vixy, threshold=0.5, kbest=120
    )
    all_summaries.append(summarise(f'D_hgb_{i}', r))
    all_res[f'D_hgb_{i}'] = r

# ----------------------------------------------------------------
# EXP E: XGBoost  (skip if not installed)
# ----------------------------------------------------------------
print('\n' + '=' * 60)
if HAS_XGB:
    print('EXP E: XGBoost')
    xgb_configs = [
        dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
             colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1.0),
        dict(n_estimators=500, max_depth=3, learning_rate=0.03, subsample=0.7,
             colsample_bytree=0.6, reg_alpha=0.5, reg_lambda=2.0),
        dict(n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8,
             colsample_bytree=0.8, reg_alpha=0.3, reg_lambda=1.5),
        dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
             colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=1.2),
    ]
    for i, cfg in enumerate(xgb_configs, 1):
        r = walk_forward(
            df_eng,
            lambda cfg=cfg: XGBClassifier(**cfg, random_state=42,
                                          eval_metric='logloss', verbosity=0),
            feat_cols_all_vixy, threshold=0.5, kbest=120
        )
        all_summaries.append(summarise(f'E_xgb_{i}', r))
        all_res[f'E_xgb_{i}'] = r
else:
    print('EXP E: SKIPPED (xgboost not installed — pip install xgboost)')

# ----------------------------------------------------------------
# EXP F: LightGBM  (skip if not installed)
# ----------------------------------------------------------------
print('\n' + '=' * 60)
if HAS_LGBM:
    print('EXP F: LightGBM')
    lgbm_configs = [
        dict(n_estimators=500, max_depth=4, learning_rate=0.03, num_leaves=15,
             subsample=0.8, colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1.0),
        dict(n_estimators=300, max_depth=3, learning_rate=0.05, num_leaves=10,
             subsample=0.7, colsample_bytree=0.6, min_child_samples=30),
        dict(n_estimators=500, max_depth=4, learning_rate=0.03, num_leaves=20,
             subsample=0.8, colsample_bytree=0.7, reg_alpha=0.3, reg_lambda=2.0,
             class_weight='balanced'),
    ]
    for i, cfg in enumerate(lgbm_configs, 1):
        r = walk_forward(
            df_eng,
            lambda cfg=cfg: LGBMClassifier(**cfg, random_state=42, verbose=-1),
            feat_cols_all_vixy, threshold=0.5, kbest=120
        )
        all_summaries.append(summarise(f'F_lgbm_{i}', r))
        all_res[f'F_lgbm_{i}'] = r
else:
    print('EXP F: SKIPPED (lightgbm not installed — pip install lightgbm)')

# ----------------------------------------------------------------
# EXP G: Calibrated LogReg  (Platt scaling)
# ----------------------------------------------------------------
print('\n' + '=' * 60)
print('EXP G: Calibrated LogReg (Platt scaling cv=3)')
# Note: CalibratedClassifierCV wraps the model; pipeline puts calibrated model at end
r = walk_forward(
    df_eng,
    lambda: CalibratedClassifierCV(
        LogisticRegression(penalty='l2', C=0.5, solver='saga', max_iter=2000, random_state=42),
        method='sigmoid', cv=3
    ),
    feat_cols_all_vixy, threshold=0.53, kbest=120
)
all_summaries.append(summarise('G_calibrated_logreg', r))
all_res['G_calibrated'] = r

# ----------------------------------------------------------------
# EXP H: Rolling 3-year window  (baseline config)
# ----------------------------------------------------------------
print('\n' + '=' * 60)
print('EXP H: Rolling 3-year training window')
r = walk_forward(
    df_eng,
    lambda: LogisticRegression(penalty='l2', C=0.5, solver='saga', max_iter=2000, random_state=42),
    feat_cols_all_vixy, threshold=0.56, kbest=120,
    rolling_years=3
)
all_summaries.append(summarise('H_rolling3yr', r))
all_res['H_rolling3yr'] = r

# EXP H2: Rolling 4-year window
r = walk_forward(
    df_eng,
    lambda: LogisticRegression(penalty='l2', C=0.5, solver='saga', max_iter=2000, random_state=42),
    feat_cols_all_vixy, threshold=0.56, kbest=120,
    rolling_years=4
)
all_summaries.append(summarise('H2_rolling4yr', r))
all_res['H2_rolling4yr'] = r

# ----------------------------------------------------------------
# EXP I: Recency-weighted training
# ----------------------------------------------------------------
print('\n' + '=' * 60)
print('EXP I: Recency-weighted LogReg (exponential decay)')

def exp_decay_weights(train_df, halflife_days=252):
    max_date  = train_df['date'].max()
    days_ago  = (max_date - train_df['date']).dt.days
    weights   = np.exp(-np.log(2) * days_ago / halflife_days)
    return weights.values

for hl in [126, 252, 504]:
    r = walk_forward(
        df_eng,
        lambda: LogisticRegression(penalty='l2', C=0.5, solver='saga', max_iter=2000, random_state=42),
        feat_cols_all_vixy, threshold=0.56, kbest=120,
        sample_weight_fn=lambda df, hl=hl: exp_decay_weights(df, hl)
    )
    all_summaries.append(summarise(f'I_recency_hl{hl}d', r))
    all_res[f'I_hl{hl}'] = r

# ----------------------------------------------------------------
# EXP J: Soft-voting ensemble of top 3 LogReg configs from v1
# ----------------------------------------------------------------
print('\n' + '=' * 60)
print('EXP J: Soft-voting ensemble  (top 3 LogReg from v1)')

TOP3_CONFIGS = [
    # (penalty, C, threshold) from v1 top performers
    ('l2', 0.5,  0.56),
    ('l1', 10.0, 0.53),
    ('l2', 3.0,  0.52),
]

def walk_forward_ensemble(df_feats, configs, feature_cols, test_years=TEST_YEARS,
                           kbest=BEST_KBEST, avg_threshold=0.53):
    """
    Soft average of predict_proba over multiple models.
    configs: list of (penalty, C, individual_threshold) — threshold is per-model (unused here),
             final decision uses avg_threshold.
    """
    results = []
    for year in test_years:
        train_mask = df_feats['date'].dt.year < year
        test_mask  = df_feats['date'].dt.year == year
        req_cols   = feature_cols + ['target_up']
        train_df   = df_feats[train_mask].dropna(subset=req_cols)
        test_df    = df_feats[test_mask].dropna(subset=req_cols)
        if len(train_df) < 200 or len(test_df) < 20:
            continue

        X_tr, y_tr = train_df[feature_cols].values, train_df['target_up'].values
        X_te, y_te = test_df[feature_cols].values,  test_df['target_up'].values

        prob_sum = np.zeros(len(y_te))
        for (pen, C, _) in configs:
            pipe = build_pipeline(
                LogisticRegression(penalty=pen, C=C, solver='saga', max_iter=2000, random_state=42),
                feature_cols, kbest
            )
            pipe.fit(X_tr, y_tr)
            prob_sum += pipe.predict_proba(X_te)[:, 1]
        avg_probs = prob_sum / len(configs)
        preds = (avg_probs >= avg_threshold).astype(int)

        acc     = accuracy_score(y_te, preds)
        bal_acc = balanced_accuracy_score(y_te, preds)
        try:
            auc = roc_auc_score(y_te, avg_probs)
        except Exception:
            auc = np.nan
        f1 = f1_score(y_te, preds, zero_division=0)
        results.append({'year': year, 'accuracy': acc, 'balanced_accuracy': bal_acc,
                        'roc_auc': auc, 'f1': f1,
                        'n_train': len(train_df), 'n_test': len(test_df)})
    return pd.DataFrame(results)

for avg_thr in [0.50, 0.52, 0.53, 0.55]:
    r = walk_forward_ensemble(df_eng, TOP3_CONFIGS, feat_cols_all_vixy,
                               avg_threshold=avg_thr)
    all_summaries.append(summarise(f'J_ensemble3_thr{avg_thr}', r))
    all_res[f'J_ens3_thr{avg_thr}'] = r

# Top 5 ensemble
TOP5_CONFIGS = TOP3_CONFIGS + [('l1', 2.0, 0.51), ('l2', 1.0, 0.53)]
r = walk_forward_ensemble(df_eng, TOP5_CONFIGS, feat_cols_all_vixy, avg_threshold=0.52)
all_summaries.append(summarise('J_ensemble5_thr0.52', r))
all_res['J_ens5'] = r

print('\nAll experiments complete!')

In [ ]:
# ============================================================
# CELL 6: SUMMARY & BEST MODEL SELECTION
# ============================================================
summary_df = pd.DataFrame(all_summaries).sort_values('bal_acc_mean', ascending=False)

print('\n' + '=' * 80)
print('FULL EXPERIMENT RANKING  (sorted by balanced_accuracy_mean)')
print('=' * 80)
print(summary_df.to_string(index=False))

# Save
summary_df.to_excel(OUTPUT_DIR / 'experiment_summary.xlsx', index=False)
print(f'\nSaved: {OUTPUT_DIR}/experiment_summary.xlsx')

# Per-year breakdown of top 3
print('\n--- Yearly detail for top 3 experiments ---')
top3_names = summary_df.head(3)['experiment'].tolist()
for nm in top3_names:
    key = [k for k in all_res if nm in k or k in nm]
    if key:
        print(f'\n{nm}')
        print(all_res[key[0]][['year','accuracy','balanced_accuracy','roc_auc','f1']].to_string(index=False))

# Identify best
best_row = summary_df.iloc[0]
print(f'\n>>> BEST EXPERIMENT: {best_row["experiment"]}')
print(f'    bal_acc_mean = {best_row["bal_acc_mean"]:.4f}')
print(f'    acc_mean     = {best_row["acc_mean"]:.4f}')
print(f'    roc_auc_mean = {best_row["roc_auc_mean"]:.4f}')

In [ ]:
# ============================================================
# CELL 7: FINAL MODEL — train on ALL data + LIVE PREDICTION
# ============================================================
#
# After reviewing Cell 6 output, set FINAL_EXP_KEY to the best
# experiment key from all_res.  Default = baseline (v1 winner).
# Change this manually if a different exp won.
#
# FINAL_EXP_KEY options (match keys in all_res dict):
#   'A_baseline'       LogReg L2 C=0.5, expanding, thr=0.56
#   'H_rolling3yr'     same model but 3-yr rolling window
#   'J_ens3_thr0.52'   3-model ensemble
#   etc.
# ============================================================

# ---- USER: update this if a different exp won ----
FINAL_PENALTY   = 'l2'
FINAL_C         = 0.5
FINAL_KBEST     = 120
FINAL_THRESHOLD = 0.56
FINAL_FEAT_COLS = feat_cols_all_vixy
USE_ENSEMBLE    = False   # set True if ensemble won; configure configs below
ENSEMBLE_CONFIGS = TOP3_CONFIGS   # used only if USE_ENSEMBLE=True
ENSEMBLE_THRESHOLD = 0.52
# --------------------------------------------------

# Train on ALL available data (up to most recent fully-labelled row)
# Target needs T+2, so last 2 rows have no label yet — drop them
train_df_full = df_eng.dropna(subset=FINAL_FEAT_COLS + ['target_up'])
print(f'Training rows: {len(train_df_full)}')
print(f'Train period:  {train_df_full.date.min().date()} to {train_df_full.date.max().date()}')

X_full = train_df_full[FINAL_FEAT_COLS].values
y_full = train_df_full['target_up'].values

if USE_ENSEMBLE:
    # Train each ensemble member separately and save all
    trained_pipes = []
    for pen, C, _ in ENSEMBLE_CONFIGS:
        pipe = build_pipeline(
            LogisticRegression(penalty=pen, C=C, solver='saga', max_iter=2000, random_state=42),
            FINAL_FEAT_COLS, FINAL_KBEST
        )
        pipe.fit(X_full, y_full)
        trained_pipes.append(pipe)
    joblib.dump({'pipes': trained_pipes, 'threshold': ENSEMBLE_THRESHOLD,
                 'feature_cols': FINAL_FEAT_COLS},
                OUTPUT_DIR / 'vixy_ensemble_model.pkl')
    print(f'Ensemble saved -> {OUTPUT_DIR}/vixy_ensemble_model.pkl')
else:
    final_pipe = build_pipeline(
        LogisticRegression(penalty=FINAL_PENALTY, C=FINAL_C, solver='saga',
                           max_iter=2000, random_state=42),
        FINAL_FEAT_COLS, FINAL_KBEST
    )
    final_pipe.fit(X_full, y_full)
    joblib.dump({'pipe': final_pipe, 'threshold': FINAL_THRESHOLD,
                 'feature_cols': FINAL_FEAT_COLS},
                OUTPUT_DIR / 'vixy_single_model.pkl')
    print(f'Model saved -> {OUTPUT_DIR}/vixy_single_model.pkl')

# ----------------------------------------------------------------
# LIVE PREDICTION  (most recent date with full features available)
# ----------------------------------------------------------------
# For live use: features are available for date T before market close.
# VIXY-derived features are already shifted by EXTRA_MARKET_LAG=1,
# so today's feature row uses yesterday's VIXY close.
# We predict VIXY direction T+2 from today.

live_df = df_eng.dropna(subset=FINAL_FEAT_COLS).sort_values('date')
live_row = live_df.tail(1)
signal_date = live_row['date'].values[0]
X_live = live_row[FINAL_FEAT_COLS].values

if USE_ENSEMBLE:
    prob_up = np.mean([p.predict_proba(X_live)[0, 1] for p in trained_pipes])
    thr_used = ENSEMBLE_THRESHOLD
else:
    prob_up  = final_pipe.predict_proba(X_live)[0, 1]
    thr_used = FINAL_THRESHOLD

direction  = 'UP'   if prob_up >= thr_used else 'DOWN'
conf_gap   = abs(prob_up - 0.5)
confidence = 'HIGH' if conf_gap > 0.10 else ('MEDIUM' if conf_gap > 0.05 else 'LOW')

t2_date = pd.Timestamp(signal_date) + pd.offsets.BDay(2)

print('\n' + '=' * 55)
print('  VIXY LIVE PREDICTION')
print('=' * 55)
print(f'  Signal date (T)  : {pd.Timestamp(signal_date).date()}')
print(f'  Forecast date    : {t2_date.date()}  (T+2 business days)')
print(f'  P(VIXY UP)       : {prob_up:.4f}  ({prob_up*100:.1f}%)')
print(f'  Threshold used   : {thr_used}')
print(f'  Direction        : {direction}')
print(f'  Confidence       : {confidence}')
print('=' * 55)

pred_record = {
    'generated_at':    datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'signal_date':     pd.Timestamp(signal_date).date(),
    'forecast_date':   t2_date.date(),
    'prob_up':         round(prob_up, 6),
    'threshold':       thr_used,
    'direction':       direction,
    'confidence':      confidence,
    'model':           'ensemble' if USE_ENSEMBLE else f'logreg_{FINAL_PENALTY}_C{FINAL_C}',
}
pred_df = pd.DataFrame([pred_record])
pred_df.to_excel(OUTPUT_DIR / 'live_prediction.xlsx', index=False)
print(f'\nPrediction saved -> {OUTPUT_DIR}/live_prediction.xlsx')

In [ ]:
# ============================================================
# CELL 8: DAILY RUNNER
# Run this cell every day to get the latest prediction.
# It re-downloads VIXY from Yahoo, rebuilds features,
# and scores the saved model.
# No retraining needed — just re-score.
# ============================================================

def run_daily_prediction(model_path=None, ensemble=False):
    """Refresh data and score today's signal."""
    # ---- reload fresh data ----
    feats_fresh = pd.read_csv(FEATS_PATH, parse_dates=['date'])
    feats_fresh = feats_fresh.sort_values('date').reset_index(drop=True)

    end_date = (datetime.today() + timedelta(days=2)).strftime('%Y-%m-%d')
    vixy_fresh = yf.download(TICKER, start='2011-01-01', end=end_date,
                              progress=False, auto_adjust=False)
    if isinstance(vixy_fresh.columns, pd.MultiIndex):
        vixy_fresh.columns = [c[0].lower() for c in vixy_fresh.columns]
    else:
        vixy_fresh.columns = [c.lower() for c in vixy_fresh.columns]
    vixy_fresh = vixy_fresh[['close']].rename(columns={'close': 'vixy_close'})
    vixy_fresh['vixy_ret']     = vixy_fresh['vixy_close'].pct_change()
    vixy_fresh['vixy_log_ret'] = np.log(vixy_fresh['vixy_close'] / vixy_fresh['vixy_close'].shift(1))
    vixy_fresh.index.name = 'date'
    vixy_fresh = vixy_fresh.reset_index()
    vixy_fresh['date'] = pd.to_datetime(vixy_fresh['date'])

    df_fresh = feats_fresh.merge(vixy_fresh, on='date', how='inner')
    df_fresh = df_fresh.sort_values('date').reset_index(drop=True)
    df_fresh_eng = add_vixy_features_decision_safe(df_fresh, BEST_LAGS, BEST_ROLLS)

    feature_cols = get_feature_cols(df_fresh_eng, 'macro_plus_all_vixy')

    # ---- load model ----
    if ensemble:
        bundle   = joblib.load(OUTPUT_DIR / 'vixy_ensemble_model.pkl')
        pipes    = bundle['pipes']
        thr_used = bundle['threshold']
        live_row = df_fresh_eng.dropna(subset=feature_cols).sort_values('date').tail(1)
        X_live   = live_row[feature_cols].values
        prob_up  = np.mean([p.predict_proba(X_live)[0, 1] for p in pipes])
    else:
        bundle   = joblib.load(OUTPUT_DIR / 'vixy_single_model.pkl')
        pipe     = bundle['pipe']
        thr_used = bundle['threshold']
        live_row = df_fresh_eng.dropna(subset=feature_cols).sort_values('date').tail(1)
        X_live   = live_row[feature_cols].values
        prob_up  = pipe.predict_proba(X_live)[0, 1]

    signal_date = live_row['date'].values[0]
    direction   = 'UP' if prob_up >= thr_used else 'DOWN'
    conf_gap    = abs(prob_up - 0.5)
    confidence  = 'HIGH' if conf_gap > 0.10 else ('MEDIUM' if conf_gap > 0.05 else 'LOW')
    t2_date     = pd.Timestamp(signal_date) + pd.offsets.BDay(2)

    print('=' * 55)
    print('  VIXY DAILY PREDICTION')
    print('=' * 55)
    print(f'  Signal date (T)  : {pd.Timestamp(signal_date).date()}')
    print(f'  Forecast date    : {t2_date.date()}  (T+2 business days)')
    print(f'  P(VIXY UP)       : {prob_up:.4f}  ({prob_up*100:.1f}%)')
    print(f'  Direction        : {direction}')
    print(f'  Confidence       : {confidence}')
    print('=' * 55)

    pred_record = {
        'generated_at':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'signal_date':   pd.Timestamp(signal_date).date(),
        'forecast_date': t2_date.date(),
        'prob_up':       round(prob_up, 6),
        'threshold':     thr_used,
        'direction':     direction,
        'confidence':    confidence,
    }

    # Append to history log
    hist_path = OUTPUT_DIR / 'prediction_history.xlsx'
    if hist_path.exists():
        hist = pd.read_excel(hist_path)
        hist = pd.concat([hist, pd.DataFrame([pred_record])], ignore_index=True)
    else:
        hist = pd.DataFrame([pred_record])
    hist.to_excel(hist_path, index=False)

    # Also overwrite today's live file
    pd.DataFrame([pred_record]).to_excel(OUTPUT_DIR / 'live_prediction.xlsx', index=False)

    return pred_record


# ---- Run it now ----
result = run_daily_prediction(ensemble=USE_ENSEMBLE)
print('\nHistory log updated.')